<!-- ---
title: Deploy any Gemma with llama.cpp on Vertex AI
type: inference
author: Alvaro Bartolome
--- -->

# Deploy any Gemma with llama.cpp on Vertex AI

This example shows how to deploy [Gemma](https://deepmind.google/models/gemma/) models from [Google DeepMind](https://deepmind.google/) with [llama.cpp](https://github.com/ggml-org/llama.cpp) on Vertex AI. It covers four models: [Gemma 3 12B IT](https://huggingface.co/unsloth/gemma-3-12b-it-GGUF), a multimodal model for text and image understanding with a 128K-token context window; [Gemma 3n E4B IT](https://huggingface.co/unsloth/gemma-3n-E4B-it-GGUF), an efficient model with selective parameter activation designed for low-resource deployment; [FunctionGemma 270M IT](https://huggingface.co/unsloth/functiongemma-270m-it-GGUF), a compact model fine-tuned for function and tool calling; and [TranslateGemma 4B IT](https://huggingface.co/mradermacher/translategemma-4b-it-GGUF), a model fine-tuned for neural machine translation across 55 languages.

![Deploy any Gemma with llama.cpp on Vertex AI](...)

## Setup and Requirements

- You have a Google Cloud account and you are logged into the Cloud Console.
- You have the necessary permissions on Google Cloud to:
    - Register and deploy models on Vertex AI.
    - Create or push artifacts to Artifact Registry.
- You have a Hugging Face Hub account, and have accepted the gating on the Google Gemma models.
- You have Docker installed locally.
- You have Python 3.11+ with `pip` installed locally (recommended via `uv`).
    - You have installed `google-cloud-aiplatform`.

## Enable APIs and Set environment variables

### Enable APIs

Before you start running the `gcloud` commands, you need to make sure that the Vertex AI Platform, Compute, and Artifact Registry APIs are enabled on your Google Cloud account.

In [ ]:
!gcloud services enable \
    aiplatform.googleapis.com \
    compute.googleapis.com \
    container.googleapis.com \
    containerregistry.googleapis.com

### Set environment variables

Then, you can set the following environment variables to be used through the example:

In [ ]:
%env PROJECT_ID gcp-partnership-412108
%env LOCATION us-central1

> [!NOTE]
> Given that the official Google Cloud DLC for llama.cpp is not yet live, you'll need to pull and push it yourself to an Artifact Registry on Google Cloud since Vertex AI will expect the container to be pulled from the Artifact Registry, hence other registries as Docker or GitHub won't work.

In [ ]:
!docker pull ghcr.io/ggml-org/llama.cpp:server-cuda-b8477
!docker tag ghcr.io/ggml-org/llama.cpp:server-cuda-b8477 $LOCATION-docker.pkg.dev/$PROJECT_ID/vertex-model-garden/hf-llama-cpp-server.cu131.ubuntu2404:b8477
!gcloud auth configure-docker $LOCATION-docker.pkg.dev
!docker push $LOCATION-docker.pkg.dev/$PROJECT_ID/vertex-model-garden/hf-llama-cpp-server.cu131.ubuntu2404:b8477

In [ ]:
%env CONTAINER_URI $LOCATION-docker.pkg.dev/$PROJECT_ID/vertex-model-garden/hf-llama-cpp-server.cu131.ubuntu2404:b8477

## Register + Deploy

Run the cells for the model you want to deploy. Each section is independent; execute only the one that corresponds to your chosen model.

In [ ]:
%pip install google-cloud-aiplatform --quiet --upgrade

In [ ]:
import os
from google.cloud import aiplatform

aiplatform.init(project=os.getenv("PROJECT_ID"), location=os.getenv("LOCATION"))

<!-- hfoptions id="model" -->
<!-- hfoption id="Gemma 3 12B IT" -->

Gemma is a family of lightweight, state-of-the-art open models from Google, built from the same research and technology used to create the Gemini models. Gemma 3 models are multimodal, handling text and image input and generating text output, with open weights for both pre-trained variants and instruction-tuned variants. Gemma 3 has a large, 128K context window, multilingual support in over 140 languages, and is available in more sizes than previous versions. Gemma 3 models are well-suited for a variety of text generation and image understanding tasks, including question answering, summarization, and reasoning. Their relatively small size makes it possible to deploy them in environments with limited resources such as laptops, desktops or your own cloud infrastructure, democratizing access to state of the art AI models and helping foster innovation for everyone.

In [ ]:
model = aiplatform.Model.upload(
    display_name="google--gemma-3-12b-it",
    serving_container_image_uri=os.getenv("CONTAINER_URI"),
    serving_container_ports=[8080],
    serving_container_health_route="/health",
    serving_container_invoke_route_prefix="/*",
    serving_container_environment_variables={
        "LLAMA_ARG_HF_REPO": "unsloth/gemma-3-12b-it-GGUF:Q4_K_M",
        "LLAMA_ARG_N_GPU_LAYERS": "-1",
        "LLAMA_ARG_FLASH_ATTN": "auto",
        "LLAMA_ARG_CTX_SIZE": "32768",
        "LLAMA_ARG_N_PREDICT": "-1",
        "LLAMA_ARG_CONTEXT_SHIFT": "false",
        "LLAMA_ARG_PORT": "8080",
        "LLAMA_ARG_JINJA": "true",
    },
    serving_container_shared_memory_size_mb=(8 * 1024),
    serving_container_deployment_timeout=7200,
)

endpoint = aiplatform.Endpoint.create(
    display_name="google--gemma-3-12b-endpoint",
    dedicated_endpoint_enabled=True,
)

model.deploy(
    endpoint=endpoint,
    machine_type="g2-standard-12",
    accelerator_type="NVIDIA_L4",
    accelerator_count=1,
    min_replica_count=1,
    max_replica_count=1,
    deploy_request_timeout=1800,
)

model.wait()

<!-- /hfoption -->
<!-- hfoption id="Gemma 3n E4B IT" -->

Gemma is a family of lightweight, state-of-the-art open models from Google, built from the same research and technology used to create the Gemini models. Gemma 3n models are designed for efficient execution on low-resource devices. They are capable of multimodal input, handling text, image, video, and audio input, and generating text outputs, with open weights for pre-trained and instruction-tuned variants. These models were trained with data in over 140 spoken languages.

Gemma 3n models use selective parameter activation technology to reduce resource requirements. This technique allows the models to operate at an effective size of 2B and 4B parameters, which is lower than the total number of parameters they contain. For more information on Gemma 3n's efficient parameter management technology, see the [Gemma 3n](https://ai.google.dev/gemma/docs/gemma-3n#parameters) page.

In [ ]:
model = aiplatform.Model.upload(
    display_name="google--gemma-3n-e4b-it",
    serving_container_image_uri=os.getenv("CONTAINER_URI"),
    serving_container_ports=[8080],
    serving_container_health_route="/health",
    serving_container_invoke_route_prefix="/*",
    serving_container_environment_variables={
        "LLAMA_ARG_HF_REPO": "unsloth/gemma-3n-E4B-it-GGUF:Q4_K_M",
        "LLAMA_ARG_N_GPU_LAYERS": "-1",
        "LLAMA_ARG_FLASH_ATTN": "auto",
        "LLAMA_ARG_CTX_SIZE": "32768",
        "LLAMA_ARG_N_PREDICT": "-1",
        "LLAMA_ARG_CONTEXT_SHIFT": "false",
        "LLAMA_ARG_PORT": "8080",
        "LLAMA_ARG_JINJA": "true",
    },
    serving_container_shared_memory_size_mb=(8 * 1024),
    serving_container_deployment_timeout=7200,
)

endpoint = aiplatform.Endpoint.create(
    display_name="google--gemma-3n-e4b-endpoint",
    dedicated_endpoint_enabled=True,
)

model.deploy(
    endpoint=endpoint,
    machine_type="g2-standard-12",
    accelerator_type="NVIDIA_L4",
    accelerator_count=1,
    min_replica_count=1,
    max_replica_count=1,
    deploy_request_timeout=1800,
)

model.wait()

<!-- /hfoption -->
<!-- hfoption id="FunctionGemma 270M IT" -->

FunctionGemma 270M IT is a remarkably compact instruction-tuned model derived from Gemma 2 and fine-tuned specifically for structured function and tool calling. Despite fitting under 300 M parameters it reliably parses user intent and emits well-formed JSON tool-call payloads, making it a great pick for lightweight agentic pipelines where latency and cost matter.

In [ ]:
model = aiplatform.Model.upload(
    display_name="google--functiongemma-270m-it",
    serving_container_image_uri=os.getenv("CONTAINER_URI"),
    serving_container_ports=[8080],
    serving_container_health_route="/health",
    serving_container_invoke_route_prefix="/*",
    serving_container_environment_variables={
        "LLAMA_ARG_HF_REPO": "unsloth/functiongemma-270m-it-GGUF:Q4_K_M",
        "LLAMA_ARG_N_GPU_LAYERS": "-1",
        "LLAMA_ARG_FLASH_ATTN": "auto",
        "LLAMA_ARG_CTX_SIZE": "32768",
        "LLAMA_ARG_N_PREDICT": "-1",
        "LLAMA_ARG_CONTEXT_SHIFT": "false",
        "LLAMA_ARG_PORT": "8080",
        "LLAMA_ARG_JINJA": "true",
    },
    serving_container_shared_memory_size_mb=(8 * 1024),
    serving_container_deployment_timeout=7200,
)

endpoint = aiplatform.Endpoint.create(
    display_name="google--functiongemma-endpoint",
    dedicated_endpoint_enabled=True,
)

model.deploy(
    endpoint=endpoint,
    machine_type="g2-standard-12",
    accelerator_type="NVIDIA_L4",
    accelerator_count=1,
    min_replica_count=1,
    max_replica_count=1,
    deploy_request_timeout=1800,
)

model.wait()

<!-- /hfoption -->
<!-- hfoption id="TranslateGemma 4B IT" -->

TranslateGemma 4B IT is a Gemma 3 4B instruction-tuned model fine-tuned for neural machine translation. It supports a wide range of language pairs through a structured chat template that accepts `source_lang_code` and `target_lang_code` keys alongside the source text, delivering high-quality translations while remaining efficiently deployable on a single GPU.

In [ ]:
model = aiplatform.Model.upload(
    display_name="google--translategemma-4b-it",
    serving_container_image_uri=os.getenv("CONTAINER_URI"),
    serving_container_ports=[8080],
    serving_container_health_route="/health",
    serving_container_invoke_route_prefix="/*",
    serving_container_environment_variables={
        "LLAMA_ARG_HF_REPO": "mradermacher/translategemma-4b-it-GGUF:Q4_K_M",
        "LLAMA_ARG_N_GPU_LAYERS": "-1",
        "LLAMA_ARG_FLASH_ATTN": "auto",
        "LLAMA_ARG_CTX_SIZE": "32768",
        "LLAMA_ARG_N_PREDICT": "-1",
        "LLAMA_ARG_CONTEXT_SHIFT": "false",
        "LLAMA_ARG_PORT": "8080",
        "LLAMA_ARG_JINJA": "true",
    },
    serving_container_shared_memory_size_mb=(8 * 1024),
    serving_container_deployment_timeout=7200,
)

endpoint = aiplatform.Endpoint.create(
    display_name="google--translategemma-endpoint",
    dedicated_endpoint_enabled=True,
)

model.deploy(
    endpoint=endpoint,
    machine_type="g2-standard-12",
    accelerator_type="NVIDIA_L4",
    accelerator_count=1,
    min_replica_count=1,
    max_replica_count=1,
    deploy_request_timeout=1800,
)

model.wait()

<!-- /hfoption -->
<!-- /hfoptions -->

## Send Requests

Once deployed, you can send requests with the OpenAI Python SDK, as the llama.cpp container exposes OpenAI-compatible routes as `/v1/chat/completions` for the Chat Completions API.

In [ ]:
%pip install google-auth httpx openai --upgrade --quiet

In [ ]:
import subprocess
from google.auth import default
from google.auth.transport.requests import Request

import openai

credentials, _ = default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
credentials.refresh(Request())

endpoint_id = endpoint.resource_name.split("/")[-1]
project_number = subprocess.check_output(
    ["gcloud", "projects", "describe", os.getenv("PROJECT_ID"), "--format=value(projectNumber)"],
    text=True,
).strip()

client = openai.OpenAI(
    base_url=(
        f"https://{endpoint_id}.{os.getenv('LOCATION')}-{project_number}"
        f".prediction.vertexai.goog/v1/projects/{os.getenv('PROJECT_ID')}"
        f"/locations/{os.getenv('LOCATION')}/endpoints/{endpoint_id}/v1"
    ),
    api_key=credentials.token,
)

<!-- hfoptions id="model" -->
<!-- hfoption id="Gemma 3 12B IT" -->

**Input:**
- Text string, such as a question, a prompt, or a document to be summarized
- Images, normalized to 896 x 896 resolution and encoded to 256 tokens each
- Total input context of 128K tokens for the 4B, 12B, and 27B sizes

**Output:**
- Generated text in response to the input, such as an answer to a question, analysis of image content, or a summary of a document
- Total output context of 8192 tokens

In [ ]:
response = client.chat.completions.create(
    model="",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant.",
        },
        {
            "role": "user",
            "content": "What is the Gemini family of models and how does it relate to the Gemma family?",
        },
    ],
    max_tokens=256,
)
print(response.choices[0].message.content)

**Streaming**

The llama.cpp server supports server-sent events (SSE) streaming out of the box. Pass `stream=True` to the `chat.completions.create` call and iterate over the returned chunks to print tokens as they are generated, which is useful for interactive applications where perceived latency matters.

In [ ]:
stream = client.chat.completions.create(
    model="",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant.",
        },
        {
            "role": "user",
            "content": "Explain the concept of gradient descent in machine learning.",
        },
    ],
    max_tokens=256,
    stream=True,
)

for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content is not None:
        print(chunk.choices[0].delta.content, end="", flush=True)
print()

**Tool Calling**

Gemma 3 12B IT supports the OpenAI tool-calling protocol through the Jinja chat template enabled at serving time (`LLAMA_ARG_JINJA=true`). Provide a list of `tools` and set `tool_choice="auto"` to let the model decide when to invoke one.

> [!TIP]
> To prevent a `tool_call` from being truncated mid-JSON, either leave `max_tokens` unset or set it large enough that the model is not cut off before the closing brace.

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather for a given location.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and country, e.g. 'Tokyo, Japan'",
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "The temperature unit to use.",
                    },
                },
                "required": ["location"],
            },
        },
    }
]

response = client.chat.completions.create(
    model="",
    messages=[{"role": "user", "content": "What's the weather like in Paris right now?"}],
    tools=tools,
    tool_choice="auto",
)

message = response.choices[0].message
print(message)
if message.tool_calls:
    for tool_call in message.tool_calls:
        print(f"\nFunction : {tool_call.function.name}")
        print(f"Arguments: {tool_call.function.arguments}")

**Image Input**

Gemma 3 12B IT is a vision-language model. Pass an image as an `image_url` content part alongside a `text` part describing the task. The example below uses the image from the official model card.

> [!NOTE]
> Image input requires a llama.cpp build that includes Gemma 3 multimodal encoder support. Verify that the container you are using ships the required encoder weights before running this cell.

In [ ]:
response = client.chat.completions.create(
    model="",
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {
                        "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/bee.jpg",
                    },
                },
                {
                    "type": "text",
                    "text": "Describe this image in detail.",
                },
            ],
        },
    ],
    max_tokens=256,
)
print(response.choices[0].message.content)
# The image is a close-up shot of a vibrant garden scene, focusing on a cluster
# of pink cosmos flowers and a busy bumblebee.

<!-- /hfoption -->
<!-- hfoption id="Gemma 3n E4B IT" -->

**Input:**
- Text string, such as a question, a prompt, or a document to be summarized
- Images, normalized to 256x256, 512x512, or 768x768 resolution and encoded to 256 tokens each
- Audio data encoded to 6.25 tokens per second from a single channel
- Total input context of 32K tokens

**Output:**
- Generated text in response to the input, such as an answer to a question, analysis of image content, or a summary of a document
- Total output length up to 32K tokens, subtracting the request input tokens

> [!NOTE]
> Currently only text is supported for GGUF inference with llama.cpp. Set temperature = 1.0, top_k = 64, top_p = 0.95, min_p = 0.0 for best results.

In [ ]:
# From the official Gemma 3n model card example:
# the candy image shows a turtle on it. The equivalent text prompt gives the same answer.
response = client.chat.completions.create(
    model="",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant.",
        },
        {
            "role": "user",
            "content": "What animal is commonly depicted on hard candy and is known for carrying its home on its back?",
        },
    ],
    max_tokens=128,
    temperature=1.0,
    top_p=0.95,
    extra_body={"top_k": 64, "min_p": 0.0},
)
print(response.choices[0].message.content)
# A turtle.

**Streaming**

The llama.cpp server supports server-sent events (SSE) streaming out of the box. Pass `stream=True` to the `chat.completions.create` call and iterate over the returned chunks to print tokens as they are generated, which is useful for interactive applications where perceived latency matters.

In [ ]:
stream = client.chat.completions.create(
    model="",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant.",
        },
        {
            "role": "user",
            "content": "Describe a bumblebee visiting flowers in a garden.",
        },
    ],
    max_tokens=256,
    temperature=1.0,
    top_p=0.95,
    extra_body={"top_k": 64, "min_p": 0.0},
    stream=True,
)

for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content is not None:
        print(chunk.choices[0].delta.content, end="", flush=True)
print()

**Tool Calling**

Gemma 3n E4B IT supports the OpenAI tool-calling protocol through the Jinja chat template enabled at serving time (`LLAMA_ARG_JINJA=true`). Provide a list of `tools` and set `tool_choice="auto"` to let the model decide when to invoke one.

> [!TIP]
> To prevent a `tool_call` from being truncated mid-JSON, either leave `max_tokens` unset or set it large enough that the model is not cut off before the closing brace.

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather for a given location.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and country, e.g. 'Tokyo, Japan'",
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "The temperature unit to use.",
                    },
                },
                "required": ["location"],
            },
        },
    }
]

response = client.chat.completions.create(
    model="",
    messages=[{"role": "user", "content": "What's the weather like in Tokyo right now?"}],
    tools=tools,
    tool_choice="auto",
    temperature=1.0,
    top_p=0.95,
    extra_body={"top_k": 64, "min_p": 0.0},
)

message = response.choices[0].message
print(message)
if message.tool_calls:
    for tool_call in message.tool_calls:
        print(f"\nFunction : {tool_call.function.name}")
        print(f"Arguments: {tool_call.function.arguments}")

<!-- /hfoption -->
<!-- hfoption id="FunctionGemma 270M IT" -->

FunctionGemma 270M IT is instruction-tuned and can handle general chat completions in addition to its primary strength of structured tool calling. The example below shows a straightforward chat completion to verify the deployment is healthy and the model is answering as expected.

In [ ]:
response = client.chat.completions.create(
    model="",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What are the key differences between supervised and unsupervised learning?"},
    ],
    max_tokens=256,
)
print(response.choices[0].message.content)

**Tool Calling**

FunctionGemma 270M IT understands the OpenAI tool-calling protocol through the Jinja chat template enabled at serving time (`LLAMA_ARG_JINJA=true`). It has been purpose-built for this task and produces compact, low-latency tool-call responses. Provide a list of `tools` and set `tool_choice="auto"` to let the model decide when to invoke one.

> [!TIP]
> To prevent a `tool_call` from being truncated mid-JSON, either leave `max_tokens` unset or set it large enough that the model is not cut off before the closing brace.

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather for a given location.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and country, e.g. 'Tokyo, Japan'",
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "The temperature unit to use.",
                    },
                },
                "required": ["location"],
            },
        },
    }
]

response = client.chat.completions.create(
    model="",
    messages=[
        {
            "role": "developer",
            "content": "You are a model that can do function calling with the following functions",
        },
        {
            "role": "user",
            "content": "What's the weather like in London?",
        },
    ],
    tools=tools,
    tool_choice="auto",
)

message = response.choices[0].message
print(message)
if message.tool_calls:
    for tool_call in message.tool_calls:
        print(f"\nFunction : {tool_call.function.name}")
        print(f"Arguments: {tool_call.function.arguments}")

<!-- /hfoption -->
<!-- hfoption id="TranslateGemma 4B IT" -->

TranslateGemma 4B IT has been fine-tuned with a custom chat template that expects `source_lang_code` and `target_lang_code` fields alongside the source text inside the `content` list. These BCP-47 language codes tell the model which language the input is written in and which language to translate it into.

**Text Translation**

Pass the source text as a `text` content part alongside the `source_lang_code` and `target_lang_code` fields.

In [ ]:
response = client.chat.completions.create(
    model="",
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "V nejhorším případě i k prasknutí čočky.",
                    "source_lang_code": "cs",
                    "target_lang_code": "en",
                }
            ],
        }
    ],
    max_tokens=128,
)
print(response.choices[0].message.content)
# In the worst case, even to the point of rupturing the lens.

**Image Translation**

Set `"type": "image"` with a `url` field pointing to the image. The model reads the text present in the image and translates it according to the provided language codes.

In [ ]:
response = client.chat.completions.create(
    model="",
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source_lang_code": "cs",
                    "target_lang_code": "de-DE",
                    "url": "https://c7.alamy.com/comp/2YAX36N/traffic-signs-in-czech-republic-pedestrian-zone-2YAX36N.jpg",
                },
            ],
        }
    ],
    max_tokens=256,
)
print(response.choices[0].message.content)

<!-- /hfoption -->
<!-- /hfoptions -->

## Clean up

To avoid inadvertent charges you should "undeploy" the model from the endpoint as otherwise you will be charged for the instance uptime even if not used, so you should either go to the Vertex AI Console at https://console.cloud.google.com/vertex-ai/online-prediction/endpoints and delete the endpoint; or directly remove those with Python as:

In [ ]:
endpoint.undeploy_all()
model.delete()